# 2. Exploratory analysis, data collection and preprocessing

This notebook covers graded Section 2: where the data comes from, what it looks like, and how we turn it into model inputs. All exploration uses the **training split only**, so nothing about the validation or test rows shapes our choices.

In [ ]:
import gzip
import sys
from pathlib import Path

root = Path.cwd() if (Path.cwd() / "src").is_dir() else Path.cwd().parent
sys.path.insert(0, str(root))

import matplotlib.pyplot as plt  # noqa: E402
import pandas as pd  # noqa: E402

from src import data  # noqa: E402

## 2.1 Context: where the data comes from

Misra, Wan and McAuley collected these transactions from Rent the Runway, a website where customers rent clothes for an occasion. After a rental, the customer reports whether the item ran **small**, **fit**, or ran **large**, and can leave a rating and a review. The authors released the data to study product size recommendation: *Decomposing Fit Semantics for Product Size Recommendation in Metric Spaces*, RecSys 2018 ([paper](https://cseweb.ucsd.edu/~jmcauley/pdfs/recsys18e.pdf), [dataset page](https://cseweb.ucsd.edu/~jmcauley/datasets.html)).

Each line is one rental: the customer, the item and the catalog size rented, the body measurements the customer reported (height, weight, bust size, body type, age), the occasion, and the feedback. The dataset page lists 192,544 transactions, 105,508 users and 5,850 items. The file has 105,571 distinct user IDs, a small gap we could not explain from the page.

In [ ]:
raw_lines = sum(1 for line in gzip.open(data.data_dir() / data.RAW_FILE_NAME, "rt") if line.strip())
df = data.load()
train, valid, test = data.split(df)
print(f"raw lines {raw_lines:,}; after dropping exact duplicates {len(df):,}")
print(f"train {len(train):,} / valid {len(valid):,} / test {len(test):,}")

## 2.2 What the training split looks like

In [ ]:
summary = pd.Series(
    {
        "transactions": len(train),
        "users": train["user_id"].nunique(),
        "items": train["item_id"].nunique(),
        "item categories": train["category"].nunique(),
        "catalog sizes": train["size"].nunique(),
    },
    name="training split",
).to_frame()
summary

In [ ]:
cols = ["weight", "bust size", "body type", "age", "height", "rented for", "rating"]
missing = train[cols].isna().mean().mul(100).round(1).rename("% missing").to_frame()
missing

Weight is missing for about one rental in six, bust size and body type for about one in ten. The other fields are almost complete.

### Plot 1: the label is imbalanced

In [ ]:
shares = train["fit"].value_counts(normalize=True).reindex(["small", "fit", "large"]).mul(100)
ax = shares.plot.bar(rot=0, color=["#4C72B0", "#999999", "#DD8452"])
ax.set(title="Fit feedback in the training split", xlabel="fit label", ylabel="% of transactions")
for i, v in enumerate(shares):
    ax.text(i, v + 1, f"{v:.1f}%", ha="center")
plt.show()

**Takeaway:** 73.8% of rentals fit, 13.4% ran small and 12.8% ran large. A model that always answers "fit" is right three times out of four and has learned nothing. **→ motivates** average AUC and macro-F1 instead of accuracy (Section 4), and class weighting when we train (Section 3).

### Plot 2: most customers rent once

In [ ]:
per_user = train["user_id"].value_counts()
per_item = train["item_id"].value_counts()
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
for ax, counts, name in zip(axes, [per_user, per_item], ["user", "item"], strict=True):
    ax.hist(counts, bins=50, log=True)
    ax.set(title=f"Transactions per {name}", xlabel=f"transactions per {name}", ylabel=f"{name}s (log scale)")
plt.tight_layout()
plt.show()
one = per_user == 1
print(f"users with one rental: {one.mean():.1%} of users, {per_user[one].sum() / len(train):.1%} of rows")
top = per_item.head(len(per_item) // 10).sum() / len(train)
print(f"median rentals per item: {per_item.median():.0f}; top 10% of items cover {top:.1%} of rows")

**Takeaway:** 70.8% of customers rent once in the training split, and those single rentals make up 41.6% of rows. A per-customer factor cannot be learned from one rental. Items are better covered: the median item has 11 rentals and the top 10% of items carry 53.6% of rows. **→ motivates** using the reported measurements alongside latent factors (they work for one-time customers), regularizing the customer terms, and relying on item terms where items are popular.

### Plot 3: catalog size carries signal, but not in a straight line

In [ ]:
parsed = data.parse(train)
labels = ["0-4", "5-8", "9-12", "13-16", "17-20", "21-30", "31+"]
buckets = pd.cut(parsed["size"], [-1, 4, 8, 12, 16, 20, 30, 60], labels=labels)
by_size = parsed.groupby(buckets, observed=True)["fit"].value_counts(normalize=True).unstack()
by_size = by_size.reindex(columns=["small", "large"]).mul(100)
ax = by_size.plot(marker="o")
ax.set(
    title="Small and large feedback by catalog size", xlabel="catalog size bucket", ylabel="% of transactions"
)
plt.show()
parsed[["size", "weight_lbs", "bust_band", "height_in"]].corr().round(2)

**Takeaway:** rentals in the smallest sizes (0 to 4) come back "large" 17.9% of the time, against about 10% for sizes 9 to 30, while "small" climbs from 7.8% to 18.5% as size grows. Above size 30 the pattern swings back. Size and weight correlate at 0.85, so the measurements mostly restate the size a customer picks: the signal sits in the *mismatch* between a body and a size. **→ motivates** keeping size as a feature and learning per-customer and per-item size offsets, as the latent fit factors in the paper do.

### Plot 4: the occasion matters

In [ ]:
by_occasion = parsed.groupby("rented_for")["fit"].value_counts(normalize=True).unstack()
common = parsed["rented_for"].value_counts() >= len(parsed) * 0.001  # skip one-off labels
by_occasion = by_occasion.loc[common[common].index].reindex(columns=["small", "large"]).mul(100)
ax = by_occasion.sort_values("large").plot.barh()
ax.set(title="Small and large feedback by occasion", xlabel="% of transactions", ylabel="rented for")
plt.show()

**Takeaway:** "everyday" rentals come back "large" 21.2% of the time, weddings only 10.7%. **→ motivates** keeping the occasion (`rented_for`) as a model input.

### Missing measurements barely change the label

In [ ]:
weight_missing = parsed["weight_lbs"].isna().map({True: "weight missing", False: "weight present"})
parsed.groupby(weight_missing)["fit"].value_counts(normalize=True).unstack().mul(100).round(1)

**Takeaway:** the label mix is almost the same whether weight is reported (73.6% fit) or not (74.5% fit). Dropping those rentals would throw away 15.5% of the data for no gain. **→ motivates** filling missing values with the training median and adding a 0/1 "was missing" column, so the model can still use the gap if it helps.

## 2.3 Preprocessing

All code lives in `src/data.py` (`load`, `split`, `parse`, `preprocess`), so every notebook and model uses the same steps. Statistics are fitted on the training split only.

Two choices matter most. First, `rating` and the review fields are written **after** the rental, so they reveal the label (a review can say "runs small"); we drop them. Second, missing values are filled, not dropped (see the takeaway above).

In [ ]:
train_p, valid_p, test_p = data.preprocess(train, valid, test)
age = pd.to_numeric(train["age"], errors="coerce")
age_out = int((age.notna() & ~age.between(*data.AGE_RANGE)).sum())
filled = int(parsed[list(data.NUMERIC)].isna().sum().sum())
unknown = int(parsed[list(data.CATEGORICAL)].isna().sum().sum())
steps = pd.DataFrame(
    [
        (
            "Drop exact duplicate lines",
            "a copy in train and one in test would leak",
            raw_lines,
            len(df),
            raw_lines - len(df),
        ),
        (
            "Split 80/10/10, seed 158",
            "test is scored once, at the end",
            len(df),
            len(train),
            len(df) - len(train),
        ),
        (
            "Drop rating and review fields",
            "written after the rental, they reveal the fit",
            len(train),
            len(train_p),
            4,
        ),
        (
            "Parse weight, height, bust size",
            "'137lbs', 5' 8\", '34d' become numbers",
            len(train),
            len(train_p),
            0,
        ),
        (
            "Age outside 15 to 90 to missing",
            "data-entry errors (ages 0 to 117)",
            len(train),
            len(train_p),
            age_out,
        ),
        (
            "Fill numeric gaps with train median",
            "keep rows; add a was-missing flag",
            len(train),
            len(train_p),
            filled,
        ),
        (
            "Missing category to 'unknown'",
            "keep rows; body type is often blank",
            len(train),
            len(train_p),
            unknown,
        ),
    ],
    columns=["step", "why", "rows before", "rows after", "values or rows affected"],
)
steps

The "values or rows affected" column counts rows for the first two steps, columns for the third, and values for the rest. Parsing changes no counts: every reported weight, height and bust size matches its expected format. The cell below checks that the table and the data agree.

In [ ]:
assert steps.loc[0, "rows after"] == len(df) == len(train) + len(valid) + len(test)
assert len(train_p) == len(train) and train_p["txn_id"].tolist() == train["txn_id"].tolist()
assert len(valid_p) == len(valid) and len(test_p) == len(test)
assert not train_p[list(data.NUMERIC)].isna().any(axis=None)
assert set(data.POST_RENTAL).isdisjoint(train_p.columns)
fill_values = parsed[list(data.NUMERIC)].median().rename("train median (fill value)").to_frame()
fill_values

The model inputs are now `size`, the five numeric measurements (`weight_lbs`, `height_in`, `bust_band`, `cup`, `age`), their five was-missing flags, and three categories (`body_type`, `category`, `rented_for`), plus the customer and item IDs for latent factor models.